# Get BDB2026 Player-Tracking Data for WRs

For all of our 2023 rookie WRs, we want to get tracking data for their player plays from the 2023 season using BDB2026 data. 

A very important piece of what we're doing here is ***determining who the CB is that's guarding them***. 

Since we want to find separation during man coverage situations, we will want to find who the CB is that's guarding them. Currently, the BDB2026 data has tracking data for all defenders on a given players. We want to determine which CB is guarding our WR of interest.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    BDB_2026_BEFORE_PASS_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Player-Time Level WR/DB Separation Data

In [ ]:
wr_separation_df = pl.read_csv(str(SEPARATION_DATA_DIR / "01_2023_Rookie_Player_Time_Level_Separation_Data.csv"))

print(f"WR Separation Data:")
print(f"    - Num Player-Plays: {wr_separation_df.select("player_play_id").n_unique()}")
print(f"    - Number of WRs: {wr_separation_df.select("wr_nfl_id").n_unique()}")
wr_separation_df

<br><br/>
#### Before Pass Data (*BDB 2026*)

In [ ]:
# ====================================
# Load Before Pass Data
# ====================================

data_dir = Path(str(BDB_2026_BEFORE_PASS_DIR))

# ------------------------------------
# Load and Combine 18 Before Pass Datasets Into 1
# ------------------------------------

before_pass_df = (
    pl.concat(
        [
            pl.scan_csv(data_dir / f"input_2023_w{i:02d}.csv")
            .with_columns(pl.lit(i).alias("week"))
            for i in range(1, 19)
        ],
        how="vertical"
    )
    .collect()
)

# ------------------------------------
# Add Game-Play and Player-Play ID
# ------------------------------------

before_pass_df = (
    before_pass_df
    .with_columns(
        # ID For Play Within A Certain Game
        game_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String), 
        # ID For Player-Play Within A Certain Game
        player_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String) + "_" + pl.col("nfl_id").cast(pl.String)
    )
)

print("Before Pass Data From BDB 2026:")
before_pass_df

<br><br/>
## Calculate Separation Metrics
***

In [ ]:
# ==================================
# What We Want To Learn About WR Separation
# ==================================

# (1) How much separation does the WR achieve?
# (2) How efficiently does the WR create separation
#     over the course of a route?
# (3) How much separation does the WR have at the
#     moment that matters -- the throw?

# ==================================
# Separation Metrics
# ==================================

# (1) How much separation does the WR achieve?
#   - Mean Separation
#   - Median Separation
#   - 90th Percentile of Separation
#   - Separation at Throw
#   - Time in Tight Coverage

# (2) How efficient does the WR create separation
#     over the course of a route?
#   - Separation gained from start to throw
#       - Pair with separation at snap
#   - Separation gained after the defender's closest point
#   - Max separation gained
#   - Separation creation rate
#       - Change in separation over an interval. Could
#         calculate over whole route or over the final
#         second before the throw

# Others
#   - Separation Consistency
#       - STD of separation over the route
#   - Late route separation
#       - Mean separation over last 1 second before throw
#   - Separation Retention
#       - Throw time separation divided by max pre-throw separation.
#         This measures how well the WR maintain their best separation



In [ ]:
# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

PLAY_COL = "player_play_id"
FRAME_COL = "frame_id"
DISTANCE_COL = "wr_cb_distance"

# Tight coverage threshold, in yards.
TIGHT_COVERAGE_THRESHOLD = 1.0

# --------------------------------------------------
# 2. Prepare frame-level separation data
# --------------------------------------------------

separation_df = (
    wr_separation_df
    .select(
        PLAY_COL,
        FRAME_COL,
        DISTANCE_COL,
    )
    .filter(
        pl.col(PLAY_COL).is_not_null()
        & pl.col(FRAME_COL).is_not_null()
        & pl.col(DISTANCE_COL).is_not_null()
    )
    .sort([PLAY_COL, FRAME_COL])
)

# Confirm there is only one row per player-play-frame.
duplicate_frames_df = (
    separation_df
    .group_by([PLAY_COL, FRAME_COL])
    .len()
    .filter(pl.col("len") > 1)
)

if duplicate_frames_df.height > 0:
    raise ValueError(
        "Found duplicate rows per player-play-frame. "
        "Resolve these before calculating separation metrics."
    )

# --------------------------------------------------
# 3. Identify the first and last observed frames
# --------------------------------------------------

separation_df = (
    separation_df
    .with_columns(
        pl.col(FRAME_COL)
        .min()
        .over(PLAY_COL)
        .alias("first_frame_id"),

        pl.col(FRAME_COL)
        .max()
        .over(PLAY_COL)
        .alias("last_frame_id"),
    )
    .with_columns(
        (
            pl.col(FRAME_COL)
            == pl.col("first_frame_id")
        ).alias("is_first_frame"),

        (
            pl.col(FRAME_COL)
            == pl.col("last_frame_id")
        ).alias("is_last_frame"),
    )
)

# The last observed frame is the throw frame.
# All observed frames are retained, including the throw frame.

# --------------------------------------------------
# 4. Metric 1: Average separation
# --------------------------------------------------

average_separation_df = (
    separation_df
    .group_by(PLAY_COL)
    .agg(
        pl.col(DISTANCE_COL)
        .mean()
        .alias("avg_separation")
    )
)

# --------------------------------------------------
# 5. Metric 2: Separation at throw
# --------------------------------------------------

separation_at_throw_df = (
    separation_df
    .filter(pl.col("is_last_frame"))
    .select(
        PLAY_COL,
        pl.col(DISTANCE_COL)
        .alias("separation_at_throw"),
    )
)

# --------------------------------------------------
# 6. Metric 3: Percentage of time in tight coverage
# --------------------------------------------------

tight_coverage_df = (
    separation_df
    .group_by(PLAY_COL)
    .agg(
        (
            (
                pl.col(DISTANCE_COL)
                < TIGHT_COVERAGE_THRESHOLD
            )
            .mean()
            * 100
        ).alias("tight_coverage_pct")
    )
)

# --------------------------------------------------
# 7. Metric 4: Separation created after closest point
# --------------------------------------------------

closest_separation_df = (
    separation_df
    .group_by(PLAY_COL)
    .agg(
        pl.col(DISTANCE_COL)
        .min()
        .alias("min_separation")
    )
)

separation_created_df = (
    separation_at_throw_df
    .join(
        closest_separation_df,
        on=PLAY_COL,
        how="left",
    )
    .with_columns(
        (
            pl.col("separation_at_throw")
            - pl.col("min_separation")
        ).alias("separation_created_after_closest")
    )
    .select(
        PLAY_COL,
        "separation_created_after_closest",
    )
)

# --------------------------------------------------
# 8. Metric 5: Maximum separation
# --------------------------------------------------

maximum_separation_df = (
    separation_df
    .group_by(PLAY_COL)
    .agg(
        pl.col(DISTANCE_COL)
        .max()
        .alias("max_separation")
    )
)

# --------------------------------------------------
# 9. Metric 6: Minimum separation
# --------------------------------------------------

minimum_separation_df = (
    separation_df
    .group_by(PLAY_COL)
    .agg(
        pl.col(DISTANCE_COL)
        .min()
        .alias("min_separation")
    )
)

# --------------------------------------------------
# 10. Metric 7: Separation gain from first frame to throw
# --------------------------------------------------

separation_at_first_frame_df = (
    separation_df
    .filter(pl.col("is_first_frame"))
    .select(
        PLAY_COL,
        pl.col(DISTANCE_COL)
        .alias("separation_at_start"),
    )
)

separation_gain_df = (
    separation_at_first_frame_df
    .join(
        separation_at_throw_df,
        on=PLAY_COL,
        how="left",
    )
    .with_columns(
        (
            pl.col("separation_at_throw")
            - pl.col("separation_at_start")
        ).alias("separation_gain_start_to_throw")
    )
    .select(
        PLAY_COL,
        "separation_at_start",
        "separation_gain_start_to_throw",
    )
)

# --------------------------------------------------
# 11. Metric 8: Separation consistency
# --------------------------------------------------

separation_consistency_df = (
    separation_df
    .group_by(PLAY_COL)
    .agg(
        pl.col(DISTANCE_COL)
        .std(ddof=0)
        .alias("separation_sd")
    )
)

# --------------------------------------------------
# 12. Metric 9: 90th percentile separation
# --------------------------------------------------

separation_90_df = (
    separation_df
    .group_by(PLAY_COL)
    .agg(
        pl.col(DISTANCE_COL)
        .quantile(quantile=0.90)
        .alias("separation_90")
    )
)

# --------------------------------------------------
# 12. Combine all eight metrics
# --------------------------------------------------

metric_dfs = [
    average_separation_df,
    separation_at_throw_df,
    tight_coverage_df,
    separation_created_df,
    maximum_separation_df,
    minimum_separation_df,
    separation_gain_df,
    separation_consistency_df,
    separation_90_df
]

wr_separation_metrics_df = metric_dfs[0]

for metric_df in metric_dfs[1:]:
    wr_separation_metrics_df = wr_separation_metrics_df.join(
        metric_df,
        on=PLAY_COL,
        how="full",
        coalesce=True,
    )

wr_separation_metrics_df = (
    wr_separation_metrics_df
    .sort(PLAY_COL)
)

# --------------------------------------------------
# 13. Inspect the results
# --------------------------------------------------

print("WR Player-Play Level Separation Metrics:")
print(f"    - Number of Player-Plays: {wr_separation_metrics_df.select("player_play_id").n_unique()}")
wr_separation_metrics_df

In [ ]:
# ======================================
# Join Player Information Onto wr_separation_metrics
# ======================================

player_id_df = before_pass_df.select("game_id", "play_id", "nfl_id", "player_play_id").unique()

wr_separation_metrics_df = (
    wr_separation_metrics_df
    .join(
        player_id_df, 
        on = "player_play_id", 
        how = "inner"
    )
    .sort("nfl_id", "game_id", "play_id")
)

# ------------------------------------
# Reorder Columns
# ------------------------------------

wr_separation_metrics_df = (
    wr_separation_metrics_df
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "player_play_id", 
        "avg_separation",
        "separation_at_throw",	
        "tight_coverage_pct",	
        "separation_created_after_closest",
        "max_separation",	
        "min_separation",	
        "separation_at_start",	
        "separation_gain_start_to_throw",	
        "separation_sd",
        "separation_90"
    )
)

wr_separation_metrics_df

<br><br/>
## Export Data
***

In [ ]:
# =================================================
# Player-Play Level WR Separtion Data
# =================================================

# This dataset has 1 row per WR player-play from 
# one of our WRs of interest. It contains several
# metrics summarizing their separation on that
# player-play

wr_separation_metrics_df.write_csv(str(SEPARATION_DATA_DIR / "02_2023_Rookie_Player_Play_Level_Separation_Data.csv"))